In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:


def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
df.shape


In [ ]:
# Task 1: Write your code here:
df.drop(columns="Order_ID")

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df = df.dropna(subset=['Delivery_Time']) # our target


In [ ]:
for col in ['Weather', 'Traffic_Level', 'Time_of_Day','Courier_Experience_yrs']:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df.shape #after dropping

In [ ]:
# Task 3: Write your code here: dublicate
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)


In [ ]:
# Task 4: Write your code here:encode

categorical_cols = df.select_dtypes(include=["object", "category"]).columns
print(categorical_cols)



In [ ]:
from sklearn.preprocessing import OneHotEncoder
########################################

df_clean = df.drop(columns="Delivery_Time")

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_array = ohe.fit_transform(df_clean[categorical_cols])

encoded_df = pd.DataFrame(encoded_array,
                           columns=ohe.get_feature_names_out(categorical_cols))

df_clean = df_clean.drop(columns=categorical_cols)

df_clean = pd.concat([df_clean.reset_index(drop=True), encoded_df], axis=1)


df_clean = df_clean.reset_index(drop=True)

df_clean.head()


In [ ]:
# Task 5: Write your code here: standered scale
from sklearn.preprocessing import StandardScaler

numerical_cols =  df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# Apply fit_transform to scale the numerical columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 6: Write your code here: imbalance
df_clean['Delivery_Time'].value_counts(normalize=True)


In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time']

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

n_splits = 5  # K=5 Folds

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Model
model = RandomForestRegressor(n_estimators=200, random_state=42)

mae_scores = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train Random Forest
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluate using MAE only
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    print("MAE:", mae)

# Print averaged MAE across folds
print("\nAverage MAE across all folds:", np.mean(mae_scores))


In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: